In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/machine-deep-learning/test_queries.csv
/kaggle/input/competitions/machine-deep-learning/sample_submission.csv
/kaggle/input/competitions/machine-deep-learning/README.md
/kaggle/input/competitions/machine-deep-learning/train_queries.csv
/kaggle/input/competitions/machine-deep-learning/knowledge_base/attendance_policy.json
/kaggle/input/competitions/machine-deep-learning/knowledge_base/exam_schedule.json
/kaggle/input/competitions/machine-deep-learning/knowledge_base/hostel.json
/kaggle/input/competitions/machine-deep-learning/knowledge_base/placement_policy.json
/kaggle/input/competitions/machine-deep-learning/knowledge_base/academic_calendar.json
/kaggle/input/competitions/machine-deep-learning/knowledge_base/holidays.json
/kaggle/input/competitions/machine-deep-learning/knowledge_base/campus_services.json
/kaggle/input/competitions/machine-deep-learning/knowledge_base/faq.json
/kaggle/input/competitions/machine-deep-learning/knowledge_base/courses.json
/kagg

# Data Loading, EDA, Data Prep.

In [2]:
import json
import pandas as pd
from pathlib import Path

# Находим папку с базой знаний
DATA_PATH = Path("/kaggle/input")

KB_PATH = next(DATA_PATH.rglob("knowledge_base"))

# Получаем путь к основной папке датасета
BASE_PATH = KB_PATH.parent

print("Папка датасета:", BASE_PATH)
print("Папка с JSON:", KB_PATH)

# Показываем все JSON-файлы
json_files = list(KB_PATH.glob("*.json"))

for file in json_files:
    print(file.name)

Папка датасета: /kaggle/input/competitions/machine-deep-learning
Папка с JSON: /kaggle/input/competitions/machine-deep-learning/knowledge_base
attendance_policy.json
exam_schedule.json
hostel.json
placement_policy.json
academic_calendar.json
holidays.json
campus_services.json
faq.json
courses.json
faculty.json


In [3]:
knowledge_base = {}

for file in json_files:
    with open(file, "r", encoding="utf-8") as f:
        knowledge_base[file.stem] = json.load(f)

print("Загруженные категории:")
print(list(knowledge_base.keys()))

Загруженные категории:
['attendance_policy', 'exam_schedule', 'hostel', 'placement_policy', 'academic_calendar', 'holidays', 'campus_services', 'faq', 'courses', 'faculty']


In [4]:
kb_list = []

for category, data in knowledge_base.items():

    df = pd.DataFrame(data)

    df["category"] = category

    kb_list.append(df)

kb_df = pd.concat(kb_list, ignore_index=True)

print("Размер базы знаний:", kb_df.shape)

display(kb_df.head(10))

Размер базы знаний: (190, 22)


,document_id,policy,category,subject,date,time,rule,event,holiday,service,...,answer,course,semester,credits,faculty,name,department,designation,subjects,email
0,A001,Minimum attendance is 75%.,attendance_policy,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,A002,Medical leave requires documents.,attendance_policy,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,A003,Attendance shortage may restrict exams.,attendance_policy,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,A004,Biometric attendance is compulsory.,attendance_policy,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,A005,Condonation is subject to approval.,attendance_policy,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,A006,Attendance updated weekly.,attendance_policy,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,A007,Lab attendance mandatory.,attendance_policy,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,A008,Proxy attendance prohibited.,attendance_policy,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,A009,Late entry marked absent.,attendance_policy,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,A010,Attendance visible in ERP.,attendance_policy,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
# Выбираем текстовые колонки
text_columns = kb_df.select_dtypes(include="object").columns.tolist()

# Исключаем служебные колонки
text_columns = [
    col for col in text_columns
    if col not in ["document_id", "category"]
]

# Объединяем содержимое колонок в один текст
kb_df["document_text"] = (
    kb_df[text_columns]
    .fillna("")
    .astype(str)
    .agg(" ".join, axis=1)
)

# Удаляем лишние пробелы
kb_df["document_text"] = kb_df["document_text"].str.strip()

# Смотрим результат
display(
    kb_df[["document_id", "category", "document_text"]].head(10)
)

,document_id,category,document_text
0,A001,attendance_policy,Minimum attendance is 75%.
1,A002,attendance_policy,Medical leave requires documents.
2,A003,attendance_policy,Attendance shortage may restrict exams.
3,A004,attendance_policy,Biometric attendance is compulsory.
4,A005,attendance_policy,Condonation is subject to approval.
5,A006,attendance_policy,Attendance updated weekly.
6,A007,attendance_policy,Lab attendance mandatory.
7,A008,attendance_policy,Proxy attendance prohibited.
8,A009,attendance_policy,Late entry marked absent.
9,A010,attendance_policy,Attendance visible in ERP.


In [6]:
kb_df["text_length"] = kb_df["document_text"].str.len()

In [7]:
print("Всего записей:", len(kb_df))

print("Уникальных document_id:", kb_df["document_id"].nunique())

print("Пропущенных document_id:", kb_df["document_id"].isna().sum())

print("Дубликатов document_id:", kb_df["document_id"].duplicated().sum())

Всего записей: 190
Уникальных document_id: 190
Пропущенных document_id: 0
Дубликатов document_id: 0


In [8]:
display(
    kb_df.groupby("category")["text_length"]
    .agg(["count", "mean", "median", "min", "max"])
    .round(1)
)

,count,mean,median,min,max
category,,,,,
academic_calendar,15,18.5,20.0,13,23
attendance_policy,10,29.8,27.0,25,39
campus_services,15,30.6,31.0,25,35
courses,20,26.4,26.5,18,37
exam_schedule,30,28.4,29.5,19,39
faculty,20,80.9,80.5,72,91
faq,50,33.5,32.0,23,52
holidays,10,9.9,10.0,4,16
hostel,10,17.8,18.0,10,26


In [9]:
train_path = BASE_PATH / "train_queries.csv"
test_path = BASE_PATH / "test_queries.csv"
submission_path = BASE_PATH / "sample_submission.csv"

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)
sample_submission = pd.read_csv(submission_path)

print("Train:", train_df.shape)
print("Test:", test_df.shape)
print("Sample submission:", sample_submission.shape)

display(train_df.head(10))
display(test_df.head(10))
display(sample_submission.head())

Train: (110, 3)
Test: (50, 2)
Sample submission: (50, 2)


,query_id,query,document_id
0,1,Who teaches Artificial Intelligence?,F001
1,2,Artificial Intelligence faculty,F001
2,3,Instructor for Artificial Intelligence,F001
3,4,Artificial Intelligence teacher,F001
4,5,Who teaches Machine Learning?,F002
5,6,Machine Learning faculty,F002
6,7,Instructor for Machine Learning,F002
7,8,Machine Learning teacher,F002
8,9,Who teaches Deep Learning?,F003
9,10,Deep Learning faculty,F003


,query_id,query
0,1001,Machine Learning faculty
1,1002,Deep Learning teacher
2,1003,Attendance required
3,1004,ERP password issue
4,1005,Library hours
5,1006,Placement eligibility
6,1007,Bonafide certificate
7,1008,Semester start date
8,1009,Hostel gate closes
9,1010,Exam form process


,query_id,document_id
0,1001,NaN
1,1002,NaN
2,1003,NaN
3,1004,NaN
4,1005,NaN


In [10]:
print("TRAIN")
print(train_df.info())
print("\nПропуски:")
print(train_df.isna().sum())
print("\nДубликаты:")
print(train_df.duplicated().sum())

TRAIN
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 110 entries, 0 to 109
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   query_id     110 non-null    int64 
 1   query        110 non-null    object
 2   document_id  110 non-null    object
dtypes: int64(1), object(2)
memory usage: 2.7+ KB
None

Пропуски:
query_id       0
query          0
document_id    0
dtype: int64

Дубликаты:
0


In [11]:
print("Количество уникальных целевых документов:",
      train_df["document_id"].nunique())

display(
    train_df["document_id"]
    .value_counts()
    .describe()
)

display(
    train_df["document_id"]
    .value_counts()
    .head(20)
)

Количество уникальных целевых документов: 29


count    29.000000
mean      3.793103
std       0.619868
min       3.000000
25%       3.000000
50%       4.000000
75%       4.000000
max       6.000000
Name: count, dtype: float64

document_id
A001    6
F002    4
F003    4
F004    4
F001    4
F006    4
F007    4
F008    4
F009    4
F010    4
F011    4
F012    4
F005    4
F013    4
F014    4
F016    4
F015    4
F017    4
F018    4
F019    4
Name: count, dtype: int64

In [12]:
kb_ids = set(kb_df["document_id"])
train_ids = set(train_df["document_id"])

print("Документы из train, отсутствующие в базе:",
      len(train_ids - kb_ids))

print("Документы базы знаний, не встречающиеся в train:",
      len(kb_ids - train_ids))

Документы из train, отсутствующие в базе: 0
Документы базы знаний, не встречающиеся в train: 161


In [13]:
kb_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 190 entries, 0 to 189
Data columns (total 24 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   document_id    190 non-null    object 
 1   policy         10 non-null     object 
 2   category       190 non-null    object 
 3   subject        30 non-null     object 
 4   date           45 non-null     object 
 5   time           30 non-null     object 
 6   rule           20 non-null     object 
 7   event          15 non-null     object 
 8   holiday        10 non-null     object 
 9   service        15 non-null     object 
 10  timing         15 non-null     object 
 11  question       50 non-null     object 
 12  answer         50 non-null     object 
 13  course         20 non-null     object 
 14  semester       20 non-null     float64
 15  credits        20 non-null     float64
 16  faculty        20 non-null     object 
 17  name           20 non-null     object 
 18  department

In [14]:
%whos

Variable            Type             Data/Info
----------------------------------------------
BASE_PATH           PosixPath        /kaggle/input/competitions/machine-deep-learning
DATA_PATH           PosixPath        /kaggle/input
KB_PATH             PosixPath        /kaggle/input/competition<...>p-learning/knowledge_base
Path                type             <class 'pathlib._local.Path'>
category            str              faculty
data                list             n=20
df                  DataFrame           document_id           <...>20@college.edu  faculty  
dirname             str              /kaggle/input/competition<...>p-learning/knowledge_base
f                   TextIOWrapper    <_io.TextIOWrapper name='<...>ode='r' encoding='utf-8'>
file                PosixPath        /kaggle/input/competition<...>owledge_base/faculty.json
filename            str              faculty.json
filenames           list             n=10
json                module           <module 'json' from '

In [15]:
kb_df.head(3).T

,0,1,2
document_id,A001,A002,A003
policy,Minimum attendance is 75%.,Medical leave requires documents.,Attendance shortage may restrict exams.
category,attendance_policy,attendance_policy,attendance_policy
subject,NaN,NaN,NaN
date,NaN,NaN,NaN
time,NaN,NaN,NaN
rule,NaN,NaN,NaN
event,NaN,NaN,NaN
holiday,NaN,NaN,NaN
service,NaN,NaN,NaN


In [16]:
kb_df.shape

(190, 24)

In [17]:
kb_df["document_id"].nunique()

190

In [18]:
kb_df["text_length"].describe()

count    190.000000
mean      32.689474
std       18.510497
min        4.000000
25%       22.000000
50%       29.000000
75%       34.750000
max       91.000000
Name: text_length, dtype: float64

# TF - IDF

In [19]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [20]:
tfidf = TfidfVectorizer(
    lowercase=True,
    ngram_range=(1, 2)
)

kb_vectors = tfidf.fit_transform(kb_df["document_text"])

In [21]:
print("Document vectors:", kb_vectors.shape)
print("Vocabulary size:", len(tfidf.vocabulary_))

Document vectors: (190, 732)
Vocabulary size: 732


In [22]:
from sklearn.model_selection import train_test_split

train_part, val_part = train_test_split(
    train_df,
    test_size=0.3,
    random_state=42,
    stratify=train_df["document_id"]
)

print("Train:", train_part.shape)
print("Validation:", val_part.shape)

Train: (77, 3)
Validation: (33, 3)


In [23]:
val_vectors = tfidf.transform(val_part["query"])

print("Validation vectors:", val_vectors.shape)

Validation vectors: (33, 732)


In [24]:
similarity_matrix = cosine_similarity(
    val_vectors,
    kb_vectors
)

print("Similarity matrix:", similarity_matrix.shape)

Similarity matrix: (33, 190)


In [25]:
import numpy as np

top_k = 5

top_indices = np.argsort(
    similarity_matrix,
    axis=1
)[:, ::-1][:, :top_k]

top_scores = np.take_along_axis(
    similarity_matrix,
    top_indices,
    axis=1
)

top_document_ids = kb_df["document_id"].values[top_indices]

In [26]:
print("Top indices:", top_indices.shape)
print("Top scores:", top_scores.shape)
print("Top document IDs:", top_document_ids.shape)

Top indices: (33, 5)
Top scores: (33, 5)
Top document IDs: (33, 5)


In [27]:
results = val_part[["query_id", "query", "document_id"]].copy()

for i in range(5):
    results[f"top_{i+1}"] = top_document_ids[:, i]
    results[f"score_{i+1}"] = top_scores[:, i]

results.head(10)

,query_id,query,document_id,top_1,score_1,top_2,score_2,top_3,score_3,top_4,score_4,top_5,score_5
56,57,Who teaches IoT?,F015,C015,0.443954,E015,0.384505,F015,0.261353,F019,0.000000,F017,0.000000
81,82,Minimum attendance required? process,A001,A001,0.529398,P001,0.275981,P004,0.270310,A007,0.142880,A006,0.137871
91,92,ERP password reset details,Q001,Q001,0.622057,Q015,0.117729,A010,0.114968,Q038,0.112812,Q006,0.102749
0,1,Who teaches Artificial Intelligence?,F001,C001,0.750973,E001,0.573812,E021,0.560986,F001,0.445470,F018,0.000000
37,38,Cyber Security faculty,F010,C010,0.682964,E010,0.535637,E030,0.525992,F010,0.427111,S014,0.184377
40,41,Who teaches NLP?,F011,C011,0.440919,E011,0.400259,F011,0.260729,F018,0.000000,F016,0.000000
26,27,Instructor for Computer Networks,F007,C007,0.740010,E007,0.558384,E027,0.548272,F007,0.436686,C012,0.176271
45,46,Computer Vision faculty,F012,C012,0.663489,E012,0.536185,F012,0.430255,C007,0.266666,F007,0.157362
41,42,NLP faculty,F011,C011,0.495338,E011,0.356285,F011,0.292909,C005,0.134196,C004,0.134196
50,51,Instructor for Software Engineering,F013,C013,0.644005,E013,0.579047,F013,0.422579,F018,0.000000,F016,0.000000


In [28]:
metrics = {}

for k in [1, 3, 5]:
    predicted = results[
        [f"top_{i}" for i in range(1, k + 1)]
    ].values

    actual = results["document_id"].values.reshape(-1, 1)

    hits = (predicted == actual).any(axis=1)

    metrics[f"Recall@{k}"] = hits.mean()

# MRR@5
reciprocal_ranks = []

for _, row in results.iterrows():
    predicted = row[[f"top_{i}" for i in range(1, 6)]].values
    actual = row["document_id"]

    ranks = np.where(predicted == actual)[0]

    reciprocal_ranks.append(
        1 / (ranks[0] + 1) if len(ranks) > 0 else 0
    )

metrics["MRR@5"] = np.mean(reciprocal_ranks)
metrics["Hit Rate@5"] = metrics["Recall@5"]

pd.Series(metrics).round(4)

Recall@1      0.1818
Recall@3      0.6061
Recall@5      1.0000
MRR@5         0.4253
Hit Rate@5    1.0000
dtype: float64

In [29]:
top5_cols = [f"top_{i}" for i in range(1, 6)]

actual = results["document_id"].to_numpy().reshape(-1, 1)
predicted = results[top5_cols].to_numpy()

top5_hits = (predicted == actual).any(axis=1)

missed = results.loc[
    ~top5_hits,
    ["query_id", "query", "document_id"]
]

total_queries = len(results)
missed_count = len(missed)

print(f"Всего запросов: {total_queries}")
print(f"Правильный документ найден в Top-5: {total_queries - missed_count}")
print(f"Правильный документ НЕ найден в Top-5: {missed_count}")
print(f"Доля пропущенных: {missed_count / total_queries * 100:.2f}%")

display(missed)

Всего запросов: 33
Правильный документ найден в Top-5: 33
Правильный документ НЕ найден в Top-5: 0
Доля пропущенных: 0.00%


,query_id,query,document_id


# Reranker

In [30]:
import importlib.util
import torch

print("sentence-transformers installed:",
      importlib.util.find_spec("sentence_transformers") is not None)

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

sentence-transformers installed: True
PyTorch version: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4


In [31]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2",
    max_length=512,
    device="cuda"
)

print("Model loaded successfully")
print("Device:", reranker.device)

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Model loaded successfully
Device: cuda:0


In [32]:
# Получаем тексты документов по их ID
document_text_by_id = kb_df.set_index("document_id")["document_text"].to_dict()

# Формируем пары query-document
candidate_pairs = []

for _, row in results.iterrows():
    for i in range(1, 6):
        document_id = row[f"top_{i}"]
        document_text = document_text_by_id[document_id]

        candidate_pairs.append((row["query"], document_text))

# Получаем scores от Cross-Encoder
reranker_scores = reranker.predict(
    candidate_pairs,
    batch_size=32,
    show_progress_bar=True
)

# Преобразуем scores в матрицу 33 x 5
reranker_scores = reranker_scores.reshape(len(results), 5)

print("Number of pairs:", len(candidate_pairs))
print("Scores shape:", reranker_scores.shape)
print("First 5 scores:", reranker_scores[0])

Batches:   0%|          | 0/6 [00:00<?, ?it/s]

Number of pairs: 165
Scores shape: (33, 5)
First 5 scores: [  1.3656176  -3.6956697   2.6800056 -10.737326  -10.606784 ]


In [33]:
# Индексы документов в порядке убывания Cross-Encoder scores
reranked_indices = np.argsort(
    reranker_scores,
    axis=1
)[:, ::-1]

# Новые document_id
reranked_document_ids = np.take_along_axis(
    top_document_ids,
    reranked_indices,
    axis=1
)

# Новые scores
reranked_scores = np.take_along_axis(
    reranker_scores,
    reranked_indices,
    axis=1
)

# Создаём таблицу результатов
reranked_results = results[
    ["query_id", "query", "document_id"]
].copy()

for i in range(5):
    reranked_results[f"top_{i+1}"] = reranked_document_ids[:, i]
    reranked_results[f"score_{i+1}"] = reranked_scores[:, i]

display(reranked_results.head(10))

,query_id,query,document_id,top_1,score_1,top_2,score_2,top_3,score_3,top_4,score_4,top_5,score_5
56,57,Who teaches IoT?,F015,F015,2.680006,C015,1.365618,E015,-3.695670,F017,-10.606784,F019,-10.737326
81,82,Minimum attendance required? process,A001,A001,4.231615,A007,-1.389332,P001,-5.516381,A006,-7.381749,P004,-9.391574
91,92,ERP password reset details,Q001,Q001,5.192236,Q015,-5.087583,Q038,-5.673825,A010,-5.987630,Q006,-7.019134
0,1,Who teaches Artificial Intelligence?,F001,F001,3.523904,C001,3.075997,E001,-2.993583,E021,-3.074381,F018,-10.637166
37,38,Cyber Security faculty,F010,C010,7.768891,F010,7.744830,E010,-2.416026,E030,-2.583119,S014,-9.588243
40,41,Who teaches NLP?,F011,F011,3.355173,C011,1.993373,E011,-3.970270,F018,-10.077580,F016,-10.415053
26,27,Instructor for Computer Networks,F007,C007,4.951515,F007,4.372042,C012,-4.849801,E007,-5.134653,E027,-5.394926
45,46,Computer Vision faculty,F012,C012,7.982162,F012,7.534253,C007,1.168223,F007,0.589921,E012,-0.342477
41,42,NLP faculty,F011,C011,7.392833,F011,7.055618,E011,-1.239391,C004,-7.073812,C005,-8.199111
50,51,Instructor for Software Engineering,F013,C013,3.881345,F013,3.795290,E013,-1.559925,F018,-9.612796,F016,-10.105057


In [34]:
metrics_reranked = {}

for k in [1, 3, 5]:
    predicted = reranked_results[
        [f"top_{i}" for i in range(1, k + 1)]
    ].values

    actual = reranked_results["document_id"].values.reshape(-1, 1)

    hits = (predicted == actual).any(axis=1)
    metrics_reranked[f"Recall@{k}"] = hits.mean()

reciprocal_ranks = []

for _, row in reranked_results.iterrows():
    predicted = row[[f"top_{i}" for i in range(1, 6)]].values
    actual = row["document_id"]

    ranks = np.where(predicted == actual)[0]

    reciprocal_ranks.append(
        1 / (ranks[0] + 1) if len(ranks) > 0 else 0
    )

metrics_reranked["MRR@5"] = np.mean(reciprocal_ranks)
metrics_reranked["Hit Rate@5"] = metrics_reranked["Recall@5"]

pd.Series(metrics_reranked).round(4)

Recall@1      0.4848
Recall@3      1.0000
Recall@5      1.0000
MRR@5         0.7424
Hit Rate@5    1.0000
dtype: float64

# Test

In [35]:
print("Test data:")
display(test_df.head())

print("Test columns:", test_df.columns.tolist())

print("Sample submission:")
display(sample_submission.head())

print("Submission columns:", sample_submission.columns.tolist())

print("Test shape:", test_df.shape)
print("Sample submission shape:", sample_submission.shape)

Test data:


,query_id,query
0,1001,Machine Learning faculty
1,1002,Deep Learning teacher
2,1003,Attendance required
3,1004,ERP password issue
4,1005,Library hours


Test columns: ['query_id', 'query']
Sample submission:


,query_id,document_id
0,1001,NaN
1,1002,NaN
2,1003,NaN
3,1004,NaN
4,1005,NaN


Submission columns: ['query_id', 'document_id']
Test shape: (50, 2)
Sample submission shape: (50, 2)


In [36]:
test_vectors = tfidf.transform(test_df["query"])

test_similarity = cosine_similarity(test_vectors, kb_vectors)

top_indices = np.argsort(-test_similarity, axis=1)[:, :5]

top_document_ids = kb_df["document_id"].values[top_indices]

print("Candidate matrix shape:", top_document_ids.shape)

Candidate matrix shape: (50, 5)


In [37]:
candidate_pairs = []

for i, row in test_df.iterrows():
    query = row["query"]

    for document_id in top_document_ids[i]:
        document_text = document_text_by_id[document_id]
        candidate_pairs.append((query, document_text))

reranker_scores = reranker.predict(
    candidate_pairs,
    batch_size=32,
    show_progress_bar=True
)

reranker_scores = reranker_scores.reshape(len(test_df), 5)

print("Scores matrix shape:", reranker_scores.shape)

Batches:   0%|          | 0/8 [00:00<?, ?it/s]

Scores matrix shape: (50, 5)


In [38]:
reranked_indices = np.argmax(reranker_scores, axis=1)

predicted_document_ids = top_document_ids[
    np.arange(len(test_df)),
    reranked_indices
]

predictions = pd.DataFrame({
    "query_id": test_df["query_id"].values,
    "document_id": predicted_document_ids
})

display(predictions.head(10))
print("Predictions shape:", predictions.shape)
print("Unique predicted documents:", predictions["document_id"].nunique())

,query_id,document_id
0,1001,C002
1,1002,C003
2,1003,A001
3,1004,Q001
4,1005,S001
5,1006,P004
6,1007,Q003
7,1008,AC001
8,1009,HS001
9,1010,Q010


Predictions shape: (50, 2)
Unique predicted documents: 49


In [39]:
submission = sample_submission[["query_id"]].merge(
    predictions,
    on="query_id",
    how="left"
)

submission.to_csv("submission.csv", index=False)

print("Submission shape:", submission.shape)
print("Missing predictions:", submission["document_id"].isna().sum())

display(submission.head(10))

Submission shape: (50, 2)
Missing predictions: 0


,query_id,document_id
0,1001,C002
1,1002,C003
2,1003,A001
3,1004,Q001
4,1005,S001
5,1006,P004
6,1007,Q003
7,1008,AC001
8,1009,HS001
9,1010,Q010
